<a href="https://colab.research.google.com/github/miriamamin1213-ux/Ablation_studies/blob/main/HecktorGPT2CN3foldCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# HECKTOR GPT-2 CLASSIFIER
# HPV Status Prediction | Compact Single-Cell Notebook
# ============================================================

# 1. IMPORT LIBRARIES, SET SEEDS AND SELECT DEVICE
import os, random, gdown
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

SEED = 42
def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. LOAD HECKTOR EXCEL DATA
DATA_FILE = "HPV2025.xlsx"
if not os.path.exists(DATA_FILE):
    downloaded = gdown.download(
        id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
        output=DATA_FILE, quiet=False
    )
    if downloaded is None or not os.path.exists(DATA_FILE):
        raise FileNotFoundError("Could not download HPV2025.xlsx.")

df = pd.read_excel(DATA_FILE)
TARGET = "HPV Status"
FEATURES = [
    "Age", "Gender", "Tobacco Consumption", "Alcohol Consumption",
    "Performance Status", "T-stage", "N-stage", "M-stage"
]
missing = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing:
    raise KeyError(f"Required Excel columns missing: {missing}")

# 3. ROBUSTLY NORMALISE TARGET AND SELECT FEATURES
def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"0", "0.0", "negative", "neg", "false", "no"}:
        return 0
    if s in {"1", "1.0", "positive", "pos", "true", "yes"}:
        return 1
    try:
        n = float(s)
        if n in (0, 1):
            return int(n)
    except (ValueError, TypeError):
        pass
    return np.nan

df["HPV_Label"] = df[TARGET].apply(normalize_target)
invalid_count = int(df["HPV_Label"].isna().sum())
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
df = df[FEATURES + ["HPV_Label"]].copy()

if df.empty:
    raise ValueError("No recognised HPV labels found; check the target column.")
counts = df["HPV_Label"].value_counts().reindex([0, 1], fill_value=0)
if (counts < 2).any():
    raise ValueError(f"Insufficient examples for both HPV classes: {counts.to_dict()}")

print("Dataset:", df.shape, "| Invalid/missing target rows dropped:", invalid_count)
print("HPV classes (0=negative, 1=positive):\n", counts)

# 4. STRATIFIED TRAIN / VALIDATION / TEST SPLIT (65/15/20)
train_val_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["HPV_Label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.1875, random_state=SEED,
    stratify=train_val_df["HPV_Label"]
)
train_df, val_df, test_df = [x.copy() for x in (train_df, val_df, test_df)]

# 5. IMPUTE MISSING FEATURES USING TRAINING DATA ONLY
fill_values = {}
for col in FEATURES:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        value = train_df[col].median()
        if pd.isna(value):
            value = 0
    else:
        modes = train_df[col].dropna().astype(str).mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[col] = value
    for split in (train_df, val_df, test_df):
        split[col] = split[col].fillna(value)
        if split[col].dtype == object:
            split[col] = split[col].astype(str).replace(
                {"": "Unknown", "nan": "Unknown", "None": "Unknown"}
            )

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)
for name, y in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: n={len(y)}, class counts={np.bincount(y, minlength=2)}")

# 6. CONVERT CLINICAL FEATURES INTO TEXT PROMPTS
def display_value(x):
    if pd.isna(x):
        return "Unknown"
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x).strip()

def format_stage(x, prefix):
    value = display_value(x)
    if value == "Unknown" or value.upper().startswith(prefix):
        return value
    try:
        n = float(value)
        if n.is_integer():
            return f"{prefix}{int(n)}"
    except ValueError:
        pass
    return f"{prefix}{value}"

def patient_to_text(row):
    labels = {
        "Age": "Age", "Gender": "Gender",
        "Tobacco Consumption": "Tobacco consumption",
        "Alcohol Consumption": "Alcohol consumption",
        "Performance Status": "Performance status",
        "T-stage": "Tumour T stage", "N-stage": "Nodal N stage",
        "M-stage": "Metastatic M stage"
    }
    parts = ["Q: Predict this patient's HPV status from the following HECKTOR clinical information."]
    for col in FEATURES:
        value = display_value(row[col])
        if col in ("T-stage", "N-stage", "M-stage"):
            value = format_stage(row[col], col[0])
        parts.append(f"{labels[col]}: {value}.")
    parts.append("Is this patient HPV positive? Answer Yes or No.\nA:")
    return " ".join(parts)

train_texts = [patient_to_text(r) for _, r in train_df.iterrows()]
val_texts = [patient_to_text(r) for _, r in val_df.iterrows()]
test_texts = [patient_to_text(r) for _, r in test_df.iterrows()]

# 7. LOAD TOKENIZER AND SET TRAINING PARAMETERS
MODEL_NAME = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in train_texts]
MAX_LENGTH = min(1024, max(32, int(np.ceil(max(lengths) / 8) * 8)))
BATCH_SIZE, EPOCHS = 64, 500
LEARNING_RATE, WEIGHT_DECAY, DROPOUT = 1e-3, 1e-4, 0.1
print("Max prompt length:", max(lengths), "| MAX_LENGTH:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE, "| Epochs:", EPOCHS)

# 8. BUILD DATASETS AND DATA LOADERS
class HPVTextDataset(Dataset):
    def __init__(self, texts, labels):
        self.items = []
        for text, label in zip(texts, labels):
            enc = tokenizer(
                text, max_length=MAX_LENGTH, padding="max_length",
                truncation=True, return_tensors="pt"
            )
            self.items.append((
                enc["input_ids"].squeeze(0),
                enc["attention_mask"].squeeze(0), int(label)
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ids, mask, label = self.items[i]
        return {"input_ids": ids, "attention_mask": mask,
                "label": torch.tensor(label, dtype=torch.long)}

train_ds = HPVTextDataset(train_texts, y_train)
val_ds = HPVTextDataset(val_texts, y_val)
test_ds = HPVTextDataset(test_texts, y_test)
loader_args = {
    "batch_size": BATCH_SIZE, "num_workers": 0,
    "pin_memory": torch.cuda.is_available()
}
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)

# 9. INITIALISE GPT-2 CLASSIFIER AND CLASS-WEIGHTED LOSS
class HPVNetGPT2Text(nn.Module):
    def __init__(self):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(MODEL_NAME)

        # Freeze the entire GPT-2 backbone
        for param in self.gpt2.parameters():
            param.requires_grad = False

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = tokenizer.pad_token_id

        h = self.gpt2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(h),
            nn.Dropout(DROPOUT),
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        out = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        ).last_hidden_state

        positions = attention_mask.sum(dim=1) - 1
        pooled = out[
            torch.arange(out.size(0), device=out.device),
            positions
        ]
        return self.classifier(pooled)


model = HPVNetGPT2Text().to(device)

class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor(
    len(y_train) / (2 * np.maximum(class_counts, 1)),
    dtype=torch.float32,
    device=device
)

def weighted_loss(logits, labels):
    losses = nn.functional.cross_entropy(
        logits, labels, reduction="none"
    )
    return (losses * class_weights[labels]).mean()

# Train only the classification head
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

# 10. VALIDATION SCORING AND THRESHOLD SELECTION
def get_scores(loader):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            logits = model(ids, mask)
            scores.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            labels.extend(batch["label"].numpy())
    return np.asarray(scores), np.asarray(labels, dtype=int)

def evaluate_loss(loader):
    model.eval()
    total, n_total = 0.0, 0
    with torch.no_grad():
        for b in loader:
            ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
            labels = b["label"].to(device)
            loss = weighted_loss(model(ids, mask), labels)
            n = len(labels); total += loss.item() * n; n_total += n
    return total / max(n_total, 1)

def choose_threshold(scores, labels):
    thresholds = np.unique(np.r_[0.0, scores, 1.0])
    values = [(float(t), balanced_accuracy_score(labels, scores >= t)) for t in thresholds]
    return max(values, key=lambda x: (x[1], -abs(x[0] - 0.5)))

# 11. TRAIN MODEL AND SAVE BEST VALIDATION CHECKPOINTS
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/HECKTOR_GPT2_Discriminative_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    "Validation_Loss": OUT_DIR / "best_model_validation_loss.pth",
    "Validation_ROC_AUC": OUT_DIR / "best_model_validation_roc_auc.pth",
    "Validation_PR_AUC": OUT_DIR / "best_model_validation_pr_auc.pth",
    "Validation_Balanced_Accuracy": OUT_DIR / "best_model_validation_balanced_accuracy.pth"
}
best = {
    "Validation_Loss": float("inf"), "Validation_ROC_AUC": -float("inf"),
    "Validation_PR_AUC": -float("inf"), "Validation_Balanced_Accuracy": -float("inf")
}
best_info, history = {}, []

for epoch in range(1, EPOCHS + 1):
    model.train()
    model.gpt2.eval()
    loss_sum, n_sum = 0.0, 0
    for b in train_loader:
        ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
        labels = b["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = weighted_loss(model(ids, mask), labels)
        loss.backward()
        optimizer.step()
        n = len(labels); loss_sum += loss.item() * n; n_sum += n

    train_loss = loss_sum / max(n_sum, 1)
    val_loss = evaluate_loss(val_loader)
    val_scores, val_labels = get_scores(val_loader)
    val_auc = roc_auc_score(val_labels, val_scores)
    val_ap = average_precision_score(val_labels, val_scores)
    threshold, val_ba = choose_threshold(val_scores, val_labels)
    metrics = {
        "Validation_Loss": val_loss, "Validation_ROC_AUC": val_auc,
        "Validation_PR_AUC": val_ap, "Validation_Balanced_Accuracy": val_ba
    }
    history.append({
        "Epoch": epoch, "Train_Loss": train_loss, **metrics,
        "Validation_Threshold": threshold
    })

    for name, value in metrics.items():
        improved = value < best[name] if name == "Validation_Loss" else value > best[name]
        if improved:
            best[name] = float(value)
            best_info[name] = {"epoch": epoch, "value": float(value), "threshold": threshold}
            torch.save(model.state_dict(), paths[name])

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss={train_loss:.4f} | "
              f"Val Loss={val_loss:.4f} | ROC-AUC={val_auc:.4f} | "
              f"PR-AUC={val_ap:.4f} | BA={val_ba:.4f} | Threshold={threshold:.4f}")

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)
pd.DataFrame([
    {"Checkpoint": k, **v, "Path": str(paths[k])} for k, v in best_info.items()
]).to_csv(OUT_DIR / "best_validation_checkpoints.csv", index=False)

# 12. EVALUATE ALL CHECKPOINTS ON HELD-OUT TEST SET
results, predictions_by_model = [], {}
for name, path in paths.items():
    model.load_state_dict(torch.load(path, map_location=device))
    scores, labels = get_scores(test_loader)
    threshold = best_info[name]["threshold"]
    preds = (scores >= threshold).astype(int)
    auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)
    ba = balanced_accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    print(f"\n=== {name} | Best epoch {best_info[name]['epoch']} | Threshold {threshold:.4f} ===")
    print(classification_report(
        labels, preds, labels=[0, 1],
        target_names=["HPV Negative", "HPV Positive"], digits=4, zero_division=0
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
    print(f"Test ROC-AUC={auc:.4f} | PR-AUC={ap:.4f} | BA={ba:.4f} | F1={f1:.4f}")

    results.append({
        "Dataset": "HECKTOR", "Model": "GPT-2 Classifier",
        "Checkpoint_Selected_By": name, "Best_Epoch": best_info[name]["epoch"],
        "Validation_Threshold": threshold, "Test_ROC_AUC": auc,
        "Test_PR_AUC": ap, "Test_Balanced_Accuracy": ba, "Test_F1": f1,
        "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1]
    })
    predictions_by_model[name] = (labels, scores, preds, threshold)
    pd.DataFrame({
        "True_Label": labels, "HPV_Positive_Score": scores,
        "Predicted_Label": preds, "Threshold": threshold
    }).to_csv(OUT_DIR / f"test_predictions_{name.lower()}.csv", index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "test_metrics_by_checkpoint.csv", index=False)

# 13. SAVE ROC AND PRECISION-RECALL CURVES
for name, (labels, scores, preds, threshold) in predictions_by_model.items():
    safe = name.lower()
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    precision, recall, _ = precision_recall_curve(labels, scores)
    auc, ap = roc_auc_score(labels, scores), average_precision_score(labels, scores)

    pd.DataFrame({"FPR": fpr, "TPR": tpr, "Threshold": roc_thresholds}).to_csv(
        OUT_DIR / f"roc_coordinates_{safe}.csv", index=False
    )
    pd.DataFrame({"Precision": precision, "Recall": recall}).to_csv(
        OUT_DIR / f"pr_coordinates_{safe}.csv", index=False
    )

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"HECKTOR ROC — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"roc_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"AP={ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"HECKTOR Precision-Recall — {name}"); plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"pr_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 14. SAVE SETTINGS AND COMPARISON RESULTS
settings = {
    "Dataset": "HECKTOR", "Model": MODEL_NAME, "Target": "HPV Status",
    "Features": ", ".join(FEATURES), "Number_of_Features": len(FEATURES),
    "Split": "65/15/20 stratified", "Seed": SEED, "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE, "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY, "Dropout": DROPOUT,
    "Max_Length": MAX_LENGTH, "Optimizer": "AdamW",
    "Class_Weighted_Loss": True, "GPT2_Backbone_Frozen": True,
    "Checkpoint_Selection": "Validation loss, ROC-AUC, PR-AUC, balanced accuracy",
    "Threshold_Selection": "Maximum validation balanced accuracy"
}
pd.DataFrame([settings]).to_csv(OUT_DIR / "settings.csv", index=False)

comparison_file = Path("model_roc_pr_auc_comparison.csv")
comparison = results_df[[
    "Dataset", "Model", "Checkpoint_Selected_By", "Best_Epoch",
    "Test_ROC_AUC", "Test_PR_AUC", "Test_Balanced_Accuracy", "Test_F1"
]]
if comparison_file.exists():
    old = pd.read_csv(comparison_file)
    combined = pd.concat([old, comparison], ignore_index=True, sort=False)
    combined = combined.drop_duplicates(
        subset=["Dataset", "Model", "Checkpoint_Selected_By"], keep="last"
    )
else:
    combined = comparison
combined.to_csv(comparison_file, index=False)

print("\nAll outputs saved to:", OUT_DIR.resolve())
print("Comparison CSV:", comparison_file.resolve())

Device: cuda
Dataset: (588, 9) | Invalid/missing target rows dropped: 138
HPV classes (0=negative, 1=positive):
 HPV_Label
0     58
1    530
Name: count, dtype: int64
Train: n=381, class counts=[ 37 344]
Validation: n=89, class counts=[ 9 80]
Test: n=118, class counts=[ 12 106]
Max prompt length: 81 | MAX_LENGTH: 88
Batch size: 64 | Epochs: 500


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 100226
Trainable parameters: 100226
Epoch 50/500 | Train Loss=0.6929 | Val Loss=0.7089 | ROC-AUC=0.7139 | PR-AUC=0.9426 | BA=0.7396 | Threshold=0.5327
Epoch 100/500 | Train Loss=0.6938 | Val Loss=0.7048 | ROC-AUC=0.7097 | PR-AUC=0.9386 | BA=0.7458 | Threshold=0.5006
Epoch 150/500 | Train Loss=0.6962 | Val Loss=0.7037 | ROC-AUC=0.7125 | PR-AUC=0.9397 | BA=0.7521 | Threshold=0.4913
Epoch 200/500 | Train Loss=0.6954 | Val Loss=0.7031 | ROC-AUC=0.7097 | PR-AUC=0.9401 | BA=0.7396 | Threshold=0.5005
Epoch 250/500 | Train Loss=0.6952 | Val Loss=0.7022 | ROC-AUC=0.7111 | PR-AUC=0.9428 | BA=0.7340 | Threshold=0.5066
Epoch 300/500 | Train Loss=0.6863 | Val Loss=0.6965 | ROC-AUC=0.7056 | PR-AUC=0.9431 | BA=0.7340 | Threshold=0.4905
Epoch 350/500 | Train Loss=0.6854 | Val Loss=0.6922 | ROC-AUC=0.7000 | PR-AUC=0.9419 | BA=0.7340 | Threshold=0.4713
Epoch 400/500 | Train Loss=0.6834 | Val Loss=0.6859 | ROC-AUC=0.7014 | PR-AUC=0.9420 | BA=0.7340 | Threshold=0.4429
Epoch 450/500 |

In [3]:

# ============================================================
# HECKTOR GPT-2 DISCRIMINATIVE (FROZEN)
# HELD-OUT TEST — 95% BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score
)

# ------------------------------------------------------------
# 1. FILE AND SETTINGS
# ------------------------------------------------------------

OUT_DIR = Path(
    "/content/drive/MyDrive/HECKTOR_GPT2_Discriminative_results"
)

PREDICTION_FILE = (
    OUT_DIR / "test_predictions_validation_pr_auc.csv"
)

N_BOOTSTRAPS = 2000
SEED = 42

if not PREDICTION_FILE.exists():
    raise FileNotFoundError(
        f"Prediction file not found: {PREDICTION_FILE}"
    )

# ------------------------------------------------------------
# 2. LOAD SELECTED CHECKPOINT TEST PREDICTIONS
# ------------------------------------------------------------

pred_df = pd.read_csv(PREDICTION_FILE)

y_true = pred_df["True_Label"].to_numpy(dtype=int)
y_score = pred_df["HPV_Positive_Score"].to_numpy(dtype=float)

# Use the threshold selected on the validation set
if "Threshold" in pred_df.columns:
    threshold = float(pred_df["Threshold"].iloc[0])
else:
    raise ValueError(
        "The saved predictions do not contain the validation threshold."
    )

y_pred = (y_score >= threshold).astype(int)

print("Test sample size:", len(y_true))
print("Class counts:", np.bincount(y_true, minlength=2))
print("Validation-selected threshold:", threshold)

# ------------------------------------------------------------
# 3. METRIC FUNCTION
# ------------------------------------------------------------

def calculate_metrics(labels, scores, predictions):

    tn, fp, fn, tp = confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "ROC_AUC": roc_auc_score(labels, scores),
        "PR_AUC": average_precision_score(labels, scores),
        "Balanced_Accuracy": balanced_accuracy_score(
            labels, predictions
        ),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1_Score": f1_score(
            labels, predictions, zero_division=0
        )
    }

# Original held-out test metrics
original_metrics = calculate_metrics(
    y_true, y_score, y_pred
)

# ------------------------------------------------------------
# 4. STRATIFIED BOOTSTRAP
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

negative_indices = np.where(y_true == 0)[0]
positive_indices = np.where(y_true == 1)[0]

bootstrap_results = {
    metric: [] for metric in original_metrics
}

for b in range(N_BOOTSTRAPS):

    boot_negative = rng.choice(
        negative_indices,
        size=len(negative_indices),
        replace=True
    )

    boot_positive = rng.choice(
        positive_indices,
        size=len(positive_indices),
        replace=True
    )

    boot_indices = np.concatenate([
        boot_negative, boot_positive
    ])

    rng.shuffle(boot_indices)

    boot_true = y_true[boot_indices]
    boot_score = y_score[boot_indices]
    boot_pred = y_pred[boot_indices]

    metrics = calculate_metrics(
        boot_true, boot_score, boot_pred
    )

    for metric, value in metrics.items():
        bootstrap_results[metric].append(value)

    if (b + 1) % 500 == 0:
        print(f"Completed {b + 1}/{N_BOOTSTRAPS}")

# ------------------------------------------------------------
# 5. CALCULATE 95% CONFIDENCE INTERVALS
# ------------------------------------------------------------

ci_rows = []

for metric, estimate in original_metrics.items():

    values = np.asarray(bootstrap_results[metric])

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    ci_rows.append({
        "Dataset": "HECKTOR",
        "Model": "GPT-2 Clinical Narrative Discriminative (Frozen)",
        "Checkpoint": "Validation_PR_AUC",
        "Selected_Epoch": 306,
        "Evaluation": "Held-out test",
        "Test_Samples": len(y_true),
        "Metric": metric,
        "Estimate": estimate,
        "CI_Lower_95": lower,
        "CI_Upper_95": upper,
        "Threshold": threshold
    })

ci_df = pd.DataFrame(ci_rows)

# ------------------------------------------------------------
# 6. SAVE CONFIDENCE INTERVALS
# ------------------------------------------------------------

ci_file = (
    OUT_DIR /
    "HECKTOR_GPT2_Discriminative_confidence_intervals.csv"
)

ci_df.to_csv(ci_file, index=False)

print("\n========== HELD-OUT TEST CONFIDENCE INTERVALS ==========")

display(
    ci_df[
        ["Metric", "Estimate", "CI_Lower_95", "CI_Upper_95"]
    ].round(4)
)

print("\nSaved to:", ci_file)


Test sample size: 118
Class counts: [ 12 106]
Validation-selected threshold: 0.5080785155296326
Completed 500/2000
Completed 1000/2000
Completed 1500/2000
Completed 2000/2000

========== HELD-OUT TEST CONFIDENCE INTERVALS ==========


,Metric,Estimate,CI_Lower_95,CI_Upper_95
0,ROC_AUC,0.8569,0.7673,0.9340
1,PR_AUC,0.9823,0.9702,0.9924
2,Balanced_Accuracy,0.6981,0.5590,0.8420
3,Sensitivity,0.8962,0.8302,0.9528
4,Specificity,0.5000,0.2500,0.7500
5,F1_Score,0.9179,0.8800,0.9519



Saved to: /content/drive/MyDrive/HECKTOR_GPT2_Discriminative_results/HECKTOR_GPT2_Discriminative_confidence_intervals.csv


In [9]:

# ============================================================
# HECKTOR GPT-2 DISCRIMINATIVE — STRATIFIED 3-FOLD CV
# Fully frozen GPT-2 backbone; classifier head only
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)


def impute_fold(train_df, val_df):
    """
    Fit imputation values on the training fold only,
    then apply them to both training and validation folds.
    """

    train_df = train_df.copy()
    val_df = val_df.copy()

    # Identify missing values in the model features
    for col in FEATURES:

        median_value = pd.to_numeric(
            train_df[col], errors="coerce"
        ).median()

        train_df[col] = pd.to_numeric(
            train_df[col], errors="coerce"
        ).fillna(median_value)

        val_df[col] = pd.to_numeric(
            val_df[col], errors="coerce"
        ).fillna(median_value)

    return train_df, val_df


# ------------------------------------------------------------
# 1. SETTINGS AND OUTPUT FOLDER
# ------------------------------------------------------------

OUT_DIR = Path(
    "/content/drive/MyDrive/HECKTOR_GPT2_Discriminative_3Fold_results"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 3
PRIMARY_CHECKPOINT = "Validation_PR_AUC"

# Keep the same experimental settings
SEED = 42
BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Output folder:", OUT_DIR)

# Check that the previous notebook sections have defined
# the required data and helper functions.
required_objects = [
    "df", "FEATURES", "tokenizer", "patient_to_text",
    "impute_fold", "HPVTextDataset", "HPVNetGPT2Text",
    "weighted_loss", "get_scores", "evaluate_loss",
    "choose_threshold", "loader_args"
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise NameError(
        f"Run the earlier setup sections first. Missing: {missing_objects}"
    )

# ------------------------------------------------------------
# 2. CREATE STRATIFIED FOLDS
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

folds = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(df[FEATURES], df["HPV_Label"]), start=1
):
    train_fold = df.iloc[train_idx].copy()
    val_fold = df.iloc[val_idx].copy()

    folds.append((train_fold, val_fold))

    print(f"\nFold {fold}")
    print("Training:", len(train_fold),
          train_fold["HPV_Label"].value_counts().sort_index().to_dict())
    print("Validation:", len(val_fold),
          val_fold["HPV_Label"].value_counts().sort_index().to_dict())

# ------------------------------------------------------------
# 3. TRAIN EACH FOLD
# ------------------------------------------------------------

checkpoint_names = [
    "Validation_Loss",
    "Validation_ROC_AUC",
    "Validation_PR_AUC",
    "Validation_Balanced_Accuracy"
]

all_fold_results = []
all_histories = []
oof_predictions = {name: [] for name in checkpoint_names}

for fold_number, (fold_train_df, fold_val_df) in enumerate(folds, start=1):

    print(f"\n{'='*55}")
    print(f"FOLD {fold_number}/{N_SPLITS}")
    print(f"{'='*55}")

    # Reproducible fold-specific initialisation
    fold_seed = SEED + fold_number
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    # Impute using training fold only
    train_df, val_df = impute_fold(fold_train_df, fold_val_df)

    y_train = train_df["HPV_Label"].to_numpy(dtype=int)
    y_val = val_df["HPV_Label"].to_numpy(dtype=int)

    print("Train class counts:", np.bincount(y_train, minlength=2))
    print("Validation class counts:", np.bincount(y_val, minlength=2))

    # Construct clinical narratives
    train_texts = [
        patient_to_text(row) for _, row in train_df.iterrows()
    ]
    val_texts = [
        patient_to_text(row) for _, row in val_df.iterrows()
    ]

    # Determine maximum sequence length from training prompts only
    lengths = [
        len(tokenizer.encode(text, add_special_tokens=False))
        for text in train_texts
    ]
    MAX_LENGTH = min(
        1024, max(32, int(np.ceil(max(lengths) / 8) * 8))
    )

    print("Maximum training prompt length:", max(lengths))
    print("MAX_LENGTH:", MAX_LENGTH)

    train_ds = HPVTextDataset(train_texts, y_train)
    val_ds = HPVTextDataset(val_texts, y_val)

    generator = torch.Generator().manual_seed(fold_seed)

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        generator=generator,
        **loader_args
    )
    val_loader = DataLoader(
        val_ds,
        shuffle=False,
        **loader_args
    )

    # Fresh GPT-2 classifier for each fold
    model = HPVNetGPT2Text().to(device)

    # Explicitly freeze every GPT-2 parameter
    for param in model.gpt2.parameters():
        param.requires_grad = False

    # Ensure classifier head remains trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    model.gpt2.eval()

    trainable_count = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    classifier_count = sum(
        p.numel() for p in model.classifier.parameters()
        if p.requires_grad
    )

    print("Total trainable parameters:", trainable_count)
    print("Trainable classifier parameters:", classifier_count)

    if trainable_count != classifier_count:
        raise RuntimeError(
            "Unexpected trainable parameters: GPT-2 may not be fully frozen."
        )

    # Class weights calculated from training fold only
    class_counts = np.bincount(y_train, minlength=2)
    class_weights = torch.tensor(
        len(y_train) / (2 * np.maximum(class_counts, 1)),
        dtype=torch.float32,
        device=device
    )

    print("Class weights [negative, positive]:",
          class_weights.cpu().numpy())

    # Optimise classifier head only
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    fold_dir = OUT_DIR / f"Fold_{fold_number}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_paths = {
        name: fold_dir / f"best_model_{name.lower()}.pth"
        for name in checkpoint_names
    }

    best_values = {
        "Validation_Loss": float("inf"),
        "Validation_ROC_AUC": -float("inf"),
        "Validation_PR_AUC": -float("inf"),
        "Validation_Balanced_Accuracy": -float("inf")
    }

    best_info = {}
    history = []

    # --------------------------------------------------------
    # 4. TRAINING
    # --------------------------------------------------------

    for epoch in range(1, EPOCHS + 1):

        model.train()

        # Keep the frozen GPT-2 backbone in evaluation mode
        model.gpt2.eval()

        train_loss_sum = 0.0
        train_sample_count = 0

        for batch in train_loader:

            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(ids, mask)

            losses = nn.functional.cross_entropy(
                logits, labels, reduction="none"
            )
            loss = (losses * class_weights[labels]).mean()

            loss.backward()
            optimizer.step()

            n = len(labels)
            train_loss_sum += loss.item() * n
            train_sample_count += n

        train_loss = train_loss_sum / max(train_sample_count, 1)

        # Validation
        val_loss = evaluate_loss(val_loader)
        val_scores, val_labels = get_scores(val_loader)

        val_roc = roc_auc_score(val_labels, val_scores)
        val_pr = average_precision_score(val_labels, val_scores)
        threshold, val_ba = choose_threshold(val_scores, val_labels)

        metrics = {
            "Validation_Loss": val_loss,
            "Validation_ROC_AUC": val_roc,
            "Validation_PR_AUC": val_pr,
            "Validation_Balanced_Accuracy": val_ba
        }

        history.append({
            "Fold": fold_number,
            "Epoch": epoch,
            "Train_Loss": train_loss,
            **metrics,
            "Validation_Threshold": threshold
        })

        # Save best checkpoint for each validation criterion
        for name, value in metrics.items():

            improved = (
                value < best_values[name]
                if name == "Validation_Loss"
                else value > best_values[name]
            )

            if improved:
                best_values[name] = float(value)

                best_info[name] = {
                    "Best_Epoch": epoch,
                    "Best_Validation_Value": float(value),
                    "Threshold": float(threshold)
                }

                torch.save(
                    model.state_dict(),
                    checkpoint_paths[name]
                )

        if epoch % 50 == 0:
            print(
                f"Fold {fold_number} | Epoch {epoch}/{EPOCHS} | "
                f"Train Loss={train_loss:.4f} | "
                f"Val Loss={val_loss:.4f} | "
                f"ROC-AUC={val_roc:.4f} | "
                f"PR-AUC={val_pr:.4f} | "
                f"BA={val_ba:.4f} | "
                f"Threshold={threshold:.4f}"
            )

    # Save fold training history
    history_df = pd.DataFrame(history)
    history_df.to_csv(
        fold_dir / "training_history.csv", index=False
    )
    all_histories.append(history_df)

    # Save checkpoint-selection details
    pd.DataFrame([
        {"Checkpoint": name, **info,
         "Checkpoint_Path": str(checkpoint_paths[name])}
        for name, info in best_info.items()
    ]).to_csv(
        fold_dir / "best_validation_checkpoints.csv",
        index=False
    )

    # --------------------------------------------------------
    # 5. EVALUATE ALL FOUR SELECTED CHECKPOINTS
    # --------------------------------------------------------

    for name in checkpoint_names:

        model.load_state_dict(
            torch.load(checkpoint_paths[name], map_location=device)
        )
        model.eval()

        scores, labels = get_scores(val_loader)

        threshold = best_info[name]["Threshold"]
        predictions = (scores >= threshold).astype(int)

        roc_auc = roc_auc_score(labels, scores)
        pr_auc = average_precision_score(labels, scores)
        ba = balanced_accuracy_score(labels, predictions)
        f1 = f1_score(labels, predictions, zero_division=0)

        cm = confusion_matrix(
            labels, predictions, labels=[0, 1]
        )

        print(
            f"\nFold {fold_number} | {name} | "
            f"Selected epoch {best_info[name]['Best_Epoch']}"
        )
        print(
            f"Validation ROC-AUC={roc_auc:.4f} | "
            f"PR-AUC={pr_auc:.4f} | "
            f"BA={ba:.4f} | F1={f1:.4f}"
        )
        print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)

        all_fold_results.append({
            "Dataset": "HECKTOR",
            "Model": "GPT-2 Clinical Narrative Discriminative (Frozen)",
            "Fold": fold_number,
            "Checkpoint_Selected_By": name,
            "Best_Epoch": best_info[name]["Best_Epoch"],
            "Validation_Threshold": threshold,
            "Validation_ROC_AUC": roc_auc,
            "Validation_PR_AUC": pr_auc,
            "Validation_Balanced_Accuracy": ba,
            "Validation_F1": f1,
            "TN": cm[0, 0],
            "FP": cm[0, 1],
            "FN": cm[1, 0],
            "TP": cm[1, 1]
        })

        # Save fold-level validation predictions
        prediction_df = pd.DataFrame({
            "Fold": fold_number,
            "True_Label": labels,
            "HPV_Positive_Score": scores,
            "Predicted_Label": predictions,
            "Threshold": threshold
        })

        prediction_df.to_csv(
            fold_dir / f"validation_predictions_{name.lower()}.csv",
            index=False
        )

        # Store for pooled out-of-fold curves
        oof_predictions[name].append(prediction_df)

    del model, optimizer, train_loader, val_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nFold {fold_number} completed.")

# ------------------------------------------------------------
# 6. SAVE INDIVIDUAL FOLD RESULTS AND MEAN ± SD
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(all_fold_results)

fold_results_df.to_csv(
    OUT_DIR / "three_fold_individual_results.csv",
    index=False
)

history_all_df = pd.concat(all_histories, ignore_index=True)
history_all_df.to_csv(
    OUT_DIR / "three_fold_training_history.csv",
    index=False
)

metric_columns = [
    "Best_Epoch",
    "Validation_Threshold",
    "Validation_ROC_AUC",
    "Validation_PR_AUC",
    "Validation_Balanced_Accuracy",
    "Validation_F1"
]

summary_rows = []

for checkpoint_name, group in fold_results_df.groupby(
    "Checkpoint_Selected_By"
):

    row = {
        "Model": "GPT-2 Clinical Narrative Discriminative (Frozen)",
        "Checkpoint_Selected_By": checkpoint_name,
        "Number_of_Folds": len(group)
    }

    for metric in metric_columns:
        row[f"{metric}_Mean"] = group[metric].mean()
        row[f"{metric}_SD"] = group[metric].std(ddof=1)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    OUT_DIR / "three_fold_mean_sd_results.csv",
    index=False
)

print("\n========== THREE-FOLD MEAN ± SD ==========")
print(summary_df.round(4).to_string(index=False))

# ------------------------------------------------------------
# 7. SAVE POOLED OUT-OF-FOLD PREDICTIONS AND CURVES
# ------------------------------------------------------------

for name in checkpoint_names:

    pooled = pd.concat(
        oof_predictions[name],
        ignore_index=True
    )

    pooled.to_csv(
        OUT_DIR / f"out_of_fold_predictions_{name.lower()}.csv",
        index=False
    )

    labels = pooled["True_Label"].to_numpy(dtype=int)
    scores = pooled["HPV_Positive_Score"].to_numpy(dtype=float)

    # Pooled out-of-fold ROC
    fpr, tpr, roc_thresholds = roc_curve(labels, scores)
    roc_auc = roc_auc_score(labels, scores)

    pd.DataFrame({
        "FPR": fpr,
        "TPR": tpr,
        "Threshold": roc_thresholds
    }).to_csv(
        OUT_DIR / f"out_of_fold_ROC_coordinates_{name.lower()}.csv",
        index=False
    )

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"HECKTOR 3-Fold Out-of-Fold ROC — {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"out_of_fold_ROC_{name.lower()}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.close()

    # Pooled out-of-fold precision-recall curve
    precision, recall, pr_thresholds = precision_recall_curve(
        labels, scores
    )
    pr_auc = average_precision_score(labels, scores)

    pd.DataFrame({
        "Precision": precision,
        "Recall": recall
    }).to_csv(
        OUT_DIR / f"out_of_fold_PR_coordinates_{name.lower()}.csv",
        index=False
    )

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"HECKTOR 3-Fold Out-of-Fold PR — {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"out_of_fold_PR_{name.lower()}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.close()

    print(
        f"{name}: pooled OOF ROC-AUC={roc_auc:.4f}, "
        f"PR-AUC={pr_auc:.4f}, n={len(labels)}"
    )

# ------------------------------------------------------------
# 8. SAVE EXPERIMENT SETTINGS
# ------------------------------------------------------------

settings = {
    "Dataset": "HECKTOR",
    "Model": "openai-community/gpt2",
    "Model_Type": "Clinical Narrative Discriminative",
    "GPT2_Backbone_Frozen": True,
    "Trainable_Component": "Classification head only",
    "Target": "HPV Status",
    "Features": ", ".join(FEATURES),
    "Number_of_Features": len(FEATURES),
    "Cross_Validation": "Stratified 3-fold",
    "Shuffle": True,
    "Seed": SEED,
    "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE,
    "Learning_Rate": LEARNING_RATE,
    "Weight_Decay": WEIGHT_DECAY,
    "Dropout": DROPOUT,
    "Optimizer": "AdamW",
    "Class_Weighted_Loss": True,
    "Checkpoint_Selection": (
        "Validation loss, ROC-AUC, PR-AUC, balanced accuracy"
    ),
    "Primary_Checkpoint_Selection": PRIMARY_CHECKPOINT,
    "Threshold_Selection": "Maximum validation balanced accuracy",
    "Output": str(OUT_DIR)
}

pd.DataFrame([settings]).to_csv(
    OUT_DIR / "settings.csv",
    index=False
)

print("\n============================================")
print("THREE-FOLD DISCRIMINATIVE EXPERIMENT COMPLETE")
print("============================================")
print("Individual fold results:",
      OUT_DIR / "three_fold_individual_results.csv")
print("Mean and SD results:",
      OUT_DIR / "three_fold_mean_sd_results.csv")
print("Out-of-fold predictions and curves:",
      OUT_DIR)


Device: cuda
Output folder: /content/drive/MyDrive/HECKTOR_GPT2_Discriminative_3Fold_results

Fold 1
Training: 392 {0: 39, 1: 353}
Validation: 196 {0: 19, 1: 177}

Fold 2
Training: 392 {0: 39, 1: 353}
Validation: 196 {0: 19, 1: 177}

Fold 3
Training: 392 {0: 38, 1: 354}
Validation: 196 {0: 20, 1: 176}

FOLD 1/3
Train class counts: [ 39 353]
Validation class counts: [ 19 177]
Maximum training prompt length: 78
MAX_LENGTH: 80


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total trainable parameters: 100226
Trainable classifier parameters: 100226
Class weights [negative, positive]: [5.025641  0.5552408]
Fold 1 | Epoch 50/500 | Train Loss=0.6969 | Val Loss=0.6886 | ROC-AUC=0.7220 | PR-AUC=0.9598 | BA=0.6737 | Threshold=0.4644
Fold 1 | Epoch 100/500 | Train Loss=0.6912 | Val Loss=0.6879 | ROC-AUC=0.7232 | PR-AUC=0.9602 | BA=0.6781 | Threshold=0.4692
Fold 1 | Epoch 150/500 | Train Loss=0.6950 | Val Loss=0.6858 | ROC-AUC=0.7339 | PR-AUC=0.9630 | BA=0.7139 | Threshold=0.5266
Fold 1 | Epoch 200/500 | Train Loss=0.7000 | Val Loss=0.6915 | ROC-AUC=0.7244 | PR-AUC=0.9620 | BA=0.6809 | Threshold=0.4501
Fold 1 | Epoch 250/500 | Train Loss=0.6944 | Val Loss=0.6886 | ROC-AUC=0.7242 | PR-AUC=0.9620 | BA=0.6885 | Threshold=0.4644
Fold 1 | Epoch 300/500 | Train Loss=0.6936 | Val Loss=0.6864 | ROC-AUC=0.7294 | PR-AUC=0.9620 | BA=0.6829 | Threshold=0.4813
Fold 1 | Epoch 350/500 | Train Loss=0.6879 | Val Loss=0.6855 | ROC-AUC=0.7267 | PR-AUC=0.9620 | BA=0.6913 | Threshold=

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total trainable parameters: 100226
Trainable classifier parameters: 100226
Class weights [negative, positive]: [5.025641  0.5552408]
Fold 2 | Epoch 50/500 | Train Loss=0.6957 | Val Loss=0.6853 | ROC-AUC=0.7095 | PR-AUC=0.9554 | BA=0.6885 | Threshold=0.4947
Fold 2 | Epoch 100/500 | Train Loss=0.6978 | Val Loss=0.6864 | ROC-AUC=0.7029 | PR-AUC=0.9564 | BA=0.6896 | Threshold=0.5347
Fold 2 | Epoch 150/500 | Train Loss=0.6913 | Val Loss=0.6857 | ROC-AUC=0.7056 | PR-AUC=0.9569 | BA=0.6952 | Threshold=0.4878
Fold 2 | Epoch 200/500 | Train Loss=0.6942 | Val Loss=0.6850 | ROC-AUC=0.7022 | PR-AUC=0.9563 | BA=0.6980 | Threshold=0.5115
Fold 2 | Epoch 250/500 | Train Loss=0.6995 | Val Loss=0.6864 | ROC-AUC=0.6858 | PR-AUC=0.9536 | BA=0.6473 | Threshold=0.5342
Fold 2 | Epoch 300/500 | Train Loss=0.6957 | Val Loss=0.6850 | ROC-AUC=0.6921 | PR-AUC=0.9545 | BA=0.6670 | Threshold=0.5093
Fold 2 | Epoch 350/500 | Train Loss=0.6947 | Val Loss=0.6850 | ROC-AUC=0.6967 | PR-AUC=0.9556 | BA=0.6726 | Threshold=

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total trainable parameters: 100226
Trainable classifier parameters: 100226
Class weights [negative, positive]: [5.1578946 0.5536723]
Fold 3 | Epoch 50/500 | Train Loss=0.6965 | Val Loss=0.7098 | ROC-AUC=0.7209 | PR-AUC=0.9592 | BA=0.7068 | Threshold=0.5076
Fold 3 | Epoch 100/500 | Train Loss=0.6921 | Val Loss=0.7103 | ROC-AUC=0.7104 | PR-AUC=0.9564 | BA=0.6847 | Threshold=0.5129
Fold 3 | Epoch 150/500 | Train Loss=0.6995 | Val Loss=0.7108 | ROC-AUC=0.7193 | PR-AUC=0.9587 | BA=0.6989 | Threshold=0.5164
Fold 3 | Epoch 200/500 | Train Loss=0.6939 | Val Loss=0.7096 | ROC-AUC=0.7236 | PR-AUC=0.9610 | BA=0.6960 | Threshold=0.5049
Fold 3 | Epoch 250/500 | Train Loss=0.6898 | Val Loss=0.7089 | ROC-AUC=0.7173 | PR-AUC=0.9590 | BA=0.6881 | Threshold=0.4804
Fold 3 | Epoch 300/500 | Train Loss=0.6919 | Val Loss=0.7091 | ROC-AUC=0.7202 | PR-AUC=0.9596 | BA=0.6881 | Threshold=0.4978
Fold 3 | Epoch 350/500 | Train Loss=0.6938 | Val Loss=0.7097 | ROC-AUC=0.7159 | PR-AUC=0.9586 | BA=0.6932 | Threshold=

In [10]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/HECKTOR_GPT2_Discriminative_3Fold_results"
)

df_oof = pd.read_csv(
    folder / "out_of_fold_predictions_validation_pr_auc.csv"
)

print("Total predictions:", len(df_oof))
print("\nClass counts:")
print(df_oof["True_Label"].value_counts().sort_index())

print("\nScore distribution by fold:")
print(
    df_oof.groupby("Fold")["HPV_Positive_Score"]
    .agg(["min", "max", "mean", "std"])
)

print("\nAUC by fold:")
from sklearn.metrics import roc_auc_score, average_precision_score

for fold in sorted(df_oof["Fold"].unique()):
    d = df_oof[df_oof["Fold"] == fold]
    print(
        f"Fold {fold}: "
        f"ROC-AUC={roc_auc_score(d['True_Label'], d['HPV_Positive_Score']):.4f}, "
        f"PR-AUC={average_precision_score(d['True_Label'], d['HPV_Positive_Score']):.4f}"
    )

Total predictions: 588

Class counts:
True_Label
0     58
1    530
Name: count, dtype: int64

Score distribution by fold:
           min       max      mean       std
Fold                                        
1     0.495431  0.496051  0.495778  0.000142
2     0.509704  0.511123  0.510523  0.000306
3     0.475978  0.476745  0.476430  0.000165

AUC by fold:
Fold 1: ROC-AUC=0.7324, PR-AUC=0.9633
Fold 2: ROC-AUC=0.7083, PR-AUC=0.9581
Fold 3: ROC-AUC=0.7281, PR-AUC=0.9619
